# Imports

In [2]:
!pip install -q albumentations

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch.nn as nn

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.get_device_name(0))
!nvidia-smi

2.10.0+cu128
12.8
Tesla T4
Mon Jun 29 14:15:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------

In [4]:
try:
    import timm
except ImportError:
    !pip install -q timm --no-deps

# Datasets

In [5]:
TRAIN_CSV = "/kaggle/input/datasets/mannyquin1/celeba-datasets/celeba_train.csv"
VAL_CSV = "/kaggle/input/datasets/mannyquin1/celeba-datasets/celeba_val.csv"
TEST_CSV = "/kaggle/input/datasets/mannyquin1/celeba-datasets/celeba_test.csv"

IMAGE_DIR = "/kaggle/input/datasets/jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba"
IMAGE_SIZE = 224

BATCH_SIZE = 32

NUM_WORKERS = 2

In [6]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(162079, 18)
(20260, 18)
(20260, 18)


# Data Loader

In [7]:
train_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

valid_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2()
])

In [8]:
class CelebADataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_dir,
        transform=None
    ):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img_path = os.path.join(
            self.image_dir,
            row["image_id"]
        )

        image = cv2.imread(img_path)

        if image is None:
            raise FileNotFoundError(
                f"Image not found: {img_path}"
            )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        if self.transform:
            image = self.transform(image=image)["image"]

        hair_color = torch.as_tensor(
            row["hair_color"],
            dtype=torch.long
        )

        texture = row["hair_texture"]

        # Unknown → ignore during training
        if texture == 2:
            texture = -1

        hair_texture = torch.as_tensor(
            texture,
            dtype=torch.long
        )

        hair_attributes = torch.as_tensor([
            row["bald"],
            row["bangs"],
            row["receding"]
        ], dtype=torch.float32)
        

        facial_hair = torch.as_tensor([
            row["mustache"],
            row["goatee"],
            row["sideburns"]
        ], dtype=torch.float32)


        accessories = torch.as_tensor([
            row["glasses"],
            row["hat"],
            row["earrings"]
        ], dtype=torch.float32)


        face_features = torch.as_tensor([
            row["oval_face"],
            row["high_cheekbones"],
            row["arched_eyebrows"],
            row["bushy_eyebrows"],
            row["big_nose"],
            row["big_lips"]
        ], dtype=torch.float32)

        return {
            "image_id": row["image_id"],

            "image": image,

            "hair_color": hair_color,

            "hair_texture": hair_texture,

            "hair_attributes": hair_attributes,

            "facial_hair": facial_hair,

            "accessories": accessories,

            "face_features": face_features

        }

In [9]:
train_dataset = CelebADataset(
    dataframe=train_df,
    image_dir=IMAGE_DIR,
    transform=train_transform
)

val_dataset = CelebADataset(
    dataframe=val_df,
    image_dir=IMAGE_DIR,
    transform=valid_transform
)

test_dataset = CelebADataset(
    dataframe=test_df,
    image_dir=IMAGE_DIR,
    transform=valid_transform
)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

162079
20260
20260


In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Model

In [11]:
MODEL_NAME = "efficientnet_b0"

HIDDEN_COLOR = 256

HIDDEN_TEXTURE = 128

HIDDEN_BINARY = 512

NUM_HAIR_COLOR = 5

NUM_HAIR_TEXTURE = 2

NUM_BINARY = 15

DROPOUT = 0.3

In [12]:
class AvatarAttributeModel(nn.Module):

    def __init__(self):

        super().__init__()

        # ----------------------------
        # Backbone
        # ----------------------------

        self.backbone = timm.create_model(

            MODEL_NAME,

            pretrained=True,

            num_classes=0,

            global_pool="avg"

        )

        feature_dim = self.backbone.num_features


        # ----------------------------
        # Heads
        # ----------------------------

        self.hair_color = nn.Sequential(

            nn.Linear(feature_dim, HIDDEN_COLOR),

            nn.BatchNorm1d(HIDDEN_COLOR),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(HIDDEN_COLOR, NUM_HAIR_COLOR)

        )

        self.hair_texture = nn.Sequential(

            nn.Linear(feature_dim,HIDDEN_TEXTURE),

            nn.BatchNorm1d(HIDDEN_TEXTURE),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(HIDDEN_TEXTURE,NUM_HAIR_TEXTURE)

        )

        self.binary = nn.Sequential(

            nn.Linear(feature_dim,HIDDEN_BINARY),

            nn.BatchNorm1d(HIDDEN_BINARY),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),
            
            nn.Linear(HIDDEN_BINARY,NUM_BINARY)

        )

    def forward(self, x):

        features = self.backbone(x)


        return {

            "hair_color":
                self.hair_color(features),

            "hair_texture":
                self.hair_texture(features),

            "binary":
                self.binary(features)

        }

In [13]:
model = AvatarAttributeModel()

model

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

AvatarAttributeModel(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (co

# Training

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

print(device)

cuda


In [ ]:
# ============================================================
# COMPUTE POSITIVE CLASS WEIGHTS
# ============================================================

binary_targets = []

for sample in train_dataset:

    target = torch.cat([
        sample["hair_attributes"],
        sample["facial_hair"],
        sample["accessories"],
        sample["face_features"],
    ])

    binary_targets.append(target)

binary_targets = torch.stack(binary_targets)

positive_count = binary_targets.sum(dim=0)
negative_count = len(binary_targets) - positive_count

pos_weight = negative_count / (positive_count + 1e-6)

print("Positive Counts:")
print(positive_count)

print("\nPositive Weights:")
print(pos_weight)

In [ ]:
hair_color_criterion = nn.CrossEntropyLoss()

hair_texture_criterion = nn.CrossEntropyLoss(
    ignore_index=-1
)

binary_criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight.to(device)
)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
from torch.amp import GradScaler

USE_AMP = torch.cuda.is_available()

scaler = GradScaler(enabled=USE_AMP)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm.auto import tqdm

def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scaler,
    device,
    hair_color_criterion,
    hair_texture_criterion,
    binary_criterion,
):

    model.train()

    # -------------------------
    # Running Statistics
    # -------------------------
    running_loss = 0.0

    hair_color_correct = 0
    hair_texture_correct = 0

    hair_texture_valid = 0
    total_samples = 0

    binary_predictions = []
    binary_targets = []

    progress_bar = tqdm(train_loader, desc="Training", leave=False)

    for batch in progress_bar:

        # -------------------------
        # Move Data to GPU
        # -------------------------
        images = batch["image"].to(device)

        hair_color = batch["hair_color"].to(device)

        hair_texture = batch["hair_texture"].to(device)

        binary = torch.cat(
            [
                batch["hair_attributes"],
                batch["facial_hair"],
                batch["accessories"],
                batch["face_features"],
            ],
            dim=1,
        ).float().to(device)

        optimizer.zero_grad()

        # -------------------------
        # Forward Pass
        # -------------------------
        with autocast(
            device_type=device.type,
            enabled=USE_AMP,
        ):

            outputs = model(images)

            hair_color_loss = hair_color_criterion(
                outputs["hair_color"],
                hair_color,
            )

            hair_texture_loss = hair_texture_criterion(
                outputs["hair_texture"],
                hair_texture,
            )

            binary_loss = binary_criterion(
                outputs["binary"],
                binary,
            )

            # Total Loss
            loss = (
                HAIR_COLOR_WEIGHT * hair_color_loss +
                HAIR_TEXTURE_WEIGHT * hair_texture_loss +
                BINARY_WEIGHT * binary_loss
            )

        # -------------------------
        # Backpropagation
        # -------------------------
        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        # -------------------------
        # Statistics
        # -------------------------
        running_loss += loss.item()

        total_samples += images.size(0)

        # Hair Color Accuracy
        hair_color_pred = outputs["hair_color"].argmax(dim=1)

        hair_color_correct += (
            hair_color_pred == hair_color
        ).sum().item()

        # Hair Texture Accuracy
        valid_mask = hair_texture != -1

        if valid_mask.sum() > 0:

            texture_pred = outputs["hair_texture"].argmax(dim=1)

            hair_texture_correct += (
                texture_pred[valid_mask]
                == hair_texture[valid_mask]
            ).sum().item()

            hair_texture_valid += valid_mask.sum().item()

        # Binary Metrics
        binary_pred = (
            torch.sigmoid(outputs["binary"]) > 0.5
        ).float()

        binary_predictions.append(
            binary_pred.cpu()
        )

        binary_targets.append(
            binary.cpu()
        )

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    # -------------------------
    # Epoch Metrics
    # -------------------------

    avg_loss = running_loss / len(train_loader)

    hair_color_acc = hair_color_correct / total_samples

    if hair_texture_valid > 0:
        hair_texture_acc = (
            hair_texture_correct / hair_texture_valid
        )
    else:
        hair_texture_acc = 0.0

    binary_predictions = (
        torch.cat(binary_predictions)
        .numpy()
    )

    binary_targets = (
        torch.cat(binary_targets)
        .numpy()
    )

    binary_precision = precision_score(
        binary_targets,
        binary_predictions,
        average="micro",
        zero_division=0,
    )

    binary_recall = recall_score(
        binary_targets,
        binary_predictions,
        average="micro",
        zero_division=0,
    )

    binary_f1 = f1_score(
        binary_targets,
        binary_predictions,
        average="micro",
        zero_division=0,
    )

    return {
        "loss": avg_loss,
        "hair_color_acc": hair_color_acc,
        "hair_texture_acc": hair_texture_acc,
        "binary_precision": binary_precision,
        "binary_recall": binary_recall,
        "binary_f1": binary_f1,
    }

In [ ]:
# ============================================================
# VALIDATE ONE EPOCH
# ============================================================

@torch.no_grad()
def validate_one_epoch(
    model,
    val_loader,
    device,
    hair_color_criterion,
    hair_texture_criterion,
    binary_criterion,
):

    model.eval()

    # -------------------------
    # Running Statistics
    # -------------------------
    running_loss = 0.0

    hair_color_correct = 0
    hair_texture_correct = 0

    hair_texture_valid = 0
    total_samples = 0

    binary_predictions = []
    binary_targets = []

    progress_bar = tqdm(val_loader, desc="Validation", leave=False)

    for batch in progress_bar:

        # -------------------------
        # Move Data to GPU
        # -------------------------
        images = batch["image"].to(device)

        hair_color = batch["hair_color"].to(device)

        hair_texture = batch["hair_texture"].to(device)

        binary = torch.cat(
            [
                batch["hair_attributes"],
                batch["facial_hair"],
                batch["accessories"],
                batch["face_features"],
            ],
            dim=1,
        ).float().to(device)

        # -------------------------
        # Forward Pass
        # -------------------------
        with autocast(
            device_type=device.type,
            enabled=USE_AMP,
        ):

            outputs = model(images)

            hair_color_loss = hair_color_criterion(
                outputs["hair_color"],
                hair_color,
            )

            hair_texture_loss = hair_texture_criterion(
                outputs["hair_texture"],
                hair_texture,
            )

            binary_loss = binary_criterion(
                outputs["binary"],
                binary,
            )

            loss = (
                HAIR_COLOR_WEIGHT * hair_color_loss
                + HAIR_TEXTURE_WEIGHT * hair_texture_loss
                + BINARY_WEIGHT * binary_loss
            )

        # -------------------------
        # Statistics
        # -------------------------
        running_loss += loss.item()

        total_samples += images.size(0)

        # Hair Color Accuracy
        hair_color_pred = outputs["hair_color"].argmax(dim=1)

        hair_color_correct += (
            hair_color_pred == hair_color
        ).sum().item()

        # Hair Texture Accuracy
        valid_mask = hair_texture != -1

        if valid_mask.sum() > 0:

            texture_pred = outputs["hair_texture"].argmax(dim=1)

            hair_texture_correct += (
                texture_pred[valid_mask]
                == hair_texture[valid_mask]
            ).sum().item()

            hair_texture_valid += valid_mask.sum().item()

        # Binary Metrics
        binary_pred = (
            torch.sigmoid(outputs["binary"]) > 0.5
        ).float()

        binary_predictions.append(
            binary_pred.cpu()
        )

        binary_targets.append(
            binary.cpu()
        )

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    # -------------------------
    # Epoch Metrics
    # -------------------------

    avg_loss = running_loss / len(val_loader)

    hair_color_acc = hair_color_correct / total_samples

    if hair_texture_valid > 0:
        hair_texture_acc = (
            hair_texture_correct / hair_texture_valid
        )
    else:
        hair_texture_acc = 0.0

    binary_predictions = (
        torch.cat(binary_predictions)
        .numpy()
    )

    binary_targets = (
        torch.cat(binary_targets)
        .numpy()
    )

    binary_precision = precision_score(
        binary_targets,
        binary_predictions,
        average="micro",
        zero_division=0,
    )

    binary_recall = recall_score(
        binary_targets,
        binary_predictions,
        average="micro",
        zero_division=0,
    )

    binary_f1 = f1_score(
        binary_targets,
        binary_predictions,
        average="micro",
        zero_division=0,
    )

    return {
        "loss": avg_loss,
        "hair_color_acc": hair_color_acc,
        "hair_texture_acc": hair_texture_acc,
        "binary_precision": binary_precision,
        "binary_recall": binary_recall,
        "binary_f1": binary_f1,
    }

In [ ]:
# ============================================================
# FREEZE / UNFREEZE BACKBONE
# ============================================================

def freeze_backbone(model):

    for param in model.backbone.parameters():
        param.requires_grad = False


def unfreeze_backbone(model):

    for param in model.backbone.parameters():
        param.requires_grad = True

In [ ]:
# ============================================================
# CHECKPOINT UTILITIES
# ============================================================

import os

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)


def save_checkpoint(
    model,
    optimizer,
    scaler,
    epoch,
    metrics,
    filename,
):
    """
    Save model checkpoint.
    """

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "metrics": metrics,
    }

    torch.save(
        checkpoint,
        os.path.join(CHECKPOINT_DIR, filename)
    )

In [ ]:
# ============================================================
# EARLY STOPPING
# ============================================================

class EarlyStopping:

    def __init__(self, patience=7):

        self.patience = patience

        self.counter = 0

        self.best_loss = float("inf")

        self.early_stop = False


    def __call__(self, val_loss):

        if val_loss < self.best_loss:

            self.best_loss = val_loss

            self.counter = 0

            return True

        else:

            self.counter += 1

            print(
                f"EarlyStopping: {self.counter}/{self.patience}"
            )

            if self.counter >= self.patience:

                self.early_stop = True

            return False

In [ ]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=30,
    eta_min=1e-6,
)

In [ ]:
freeze_backbone(model)

print("Backbone Frozen")

In [ ]:
# ============================================================
# MAIN TRAINING LOOP
# ============================================================

NUM_EPOCHS = 30

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

HAIR_COLOR_WEIGHT = 1.0
HAIR_TEXTURE_WEIGHT = 1.0
BINARY_WEIGHT = 1.0

FREEZE_EPOCHS = 5

PATIENCE = 7

history = {
    "train_loss": [],
    "val_loss": [],
    "train_hair_color_acc": [],
    "val_hair_color_acc": [],
    "train_hair_texture_acc": [],
    "val_hair_texture_acc": [],
    "train_binary_f1": [],
    "val_binary_f1": [],
}

best_val_loss = float("inf")
best_val_f1 = 0.0

print("=" * 60)
print("Starting Training...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):

    if epoch == 5:

    print("\nUnfreezing Backbone...\n")

    unfreeze_backbone(model)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=1e-4,
        weight_decay=1e-4,
    )

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print("-" * 60)

    # ==========================
    # Train
    # ==========================

    train_metrics = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
        hair_color_criterion=hair_color_criterion,
        hair_texture_criterion=hair_texture_criterion,
        binary_criterion=binary_criterion,
    )

    # ==========================
    # Validation
    # ==========================

    val_metrics = validate_one_epoch(
        model=model,
        val_loader=val_loader,
        device=device,
        hair_color_criterion=hair_color_criterion,
        hair_texture_criterion=hair_texture_criterion,
        binary_criterion=binary_criterion,
    )

    scheduler.step()

    # ==========================
    # Save History
    # ==========================

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])

    history["train_hair_color_acc"].append(
        train_metrics["hair_color_acc"]
    )
    history["val_hair_color_acc"].append(
        val_metrics["hair_color_acc"]
    )

    history["train_hair_texture_acc"].append(
        train_metrics["hair_texture_acc"]
    )
    history["val_hair_texture_acc"].append(
        val_metrics["hair_texture_acc"]
    )

    history["train_binary_f1"].append(
        train_metrics["binary_f1"]
    )
    history["val_binary_f1"].append(
        val_metrics["binary_f1"]
    )

    # ==========================
    # Print Metrics
    # ==========================

    print(
        f"Train Loss          : {train_metrics['loss']:.4f}"
    )

    print(
        f"Validation Loss     : {val_metrics['loss']:.4f}"
    )

    print(
        f"Hair Color Acc      : {val_metrics['hair_color_acc']:.4f}"
    )

    print(
        f"Hair Texture Acc    : {val_metrics['hair_texture_acc']:.4f}"
    )

    print(
        f"Binary Precision    : {val_metrics['binary_precision']:.4f}"
    )

    print(
        f"Binary Recall       : {val_metrics['binary_recall']:.4f}"
    )

    print(
        f"Binary F1           : {val_metrics['binary_f1']:.4f}"
    )

    print(
        f"Learning Rate       : {optimizer.param_groups[0]['lr']:.7f}"
    )

    # ==========================
    # Save Best Loss
    # ==========================

    if val_metrics["loss"] < best_val_loss:

        best_val_loss = val_metrics["loss"]

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            metrics=val_metrics,
            filename="best_loss.pth",
        )

        print("✅ Best Loss Model Saved")

    # ==========================
    # Save Best F1
    # ==========================

    if val_metrics["binary_f1"] > best_val_f1:

        best_val_f1 = val_metrics["binary_f1"]

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            metrics=val_metrics,
            filename="best_f1.pth",
        )

        print("✅ Best F1 Model Saved")

    # ==========================
    # Save Last Model
    # ==========================

    save_checkpoint(
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
        metrics=val_metrics,
        filename="last.pth",
    )

    # ==========================
    # Early Stopping
    # ==========================

    improved = early_stopping(
        val_metrics["loss"]
    )

    if early_stopping.early_stop:

        print("\nEarly stopping triggered.")

        break

print("\nTraining Finished.")